# Phase 3 — Winter Storm Uri Deep Dive

This notebook quantifies ERCOT load stress during February 10-20, 2021 and documents generation outage facts from ERCOT post-storm review material.

In [1]:

import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
CHROME_PATH = PROJECT_ROOT / '.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR = PROJECT_ROOT / 'outputs/charts'; CHART_DIR.mkdir(parents=True, exist_ok=True)
def apply_chart_standard(fig, title, xaxis_title, yaxis_title, source='Source: ERCOT hourly load data; ERCOT February 2021 Review'):
    fig.update_layout(title=dict(text=title, font=dict(size=18, family='Arial', color='black')), font=dict(family='Arial', size=12), xaxis_title=xaxis_title, yaxis_title=yaxis_title, plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified', margin=dict(l=70, r=50, t=90, b=80))
    fig.update_xaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14)); fig.update_yaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14))
    fig.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.14, text=source, showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
    return fig
def save_chart(fig, filename):
    html_path = CHART_DIR / f'{filename}.html'; png_path = CHART_DIR / f'{filename}.png'
    fig.write_html(html_path); print(f'Saved: {html_path.relative_to(PROJECT_ROOT)}')
    subprocess.run([str(CHROME_PATH), '--headless=new', '--disable-gpu', '--no-sandbox', '--disable-dev-shm-usage', f'--screenshot={png_path}', '--window-size=1200,700', f'file://{html_path}'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(f'Saved: {png_path.relative_to(PROJECT_ROOT)}')
uri_df = pd.read_csv(PROJECT_ROOT / 'data/processed/uri_analysis.csv', parse_dates=['datetime'])
print(f'Loaded Uri analysis dataset: {uri_df.shape}')
print(uri_df.head(10).to_string(index=False))
print('Chart export method: Plotly HTML + approved headless Chrome PNG screenshot')


Loaded Uri analysis dataset: (264, 12)
           datetime  year  month  day  hour    demand_mw  baseline_load_mw  demand_surge_mw  outage_period_flag  peak_demand_in_period  min_demand_in_period  avg_baseline
2021-02-10 00:00:00  2021      2   10     0 47723.784739      37135.696854     10588.087885                   0           69692.485103           33953.36042  39539.965559
2021-02-10 01:00:00  2021      2   10     1 40520.626171      35786.848655      4733.777516                   0           69692.485103           33953.36042  39539.965559
2021-02-10 02:00:00  2021      2   10     2 39990.922868      35100.176278      4890.746590                   0           69692.485103           33953.36042  39539.965559
2021-02-10 03:00:00  2021      2   10     3 39947.716182      34877.255225      5070.460957                   0           69692.485103           33953.36042  39539.965559
2021-02-10 04:00:00  2021      2   10     4 40278.917793      35047.521300      5231.396493               

In [2]:

print('WINTER STORM URI — COMPLETE ANALYSIS')
print('=' * 60)
uri = uri_df.copy(); peak_idx = uri['demand_mw'].idxmax()
print(f"Peak demand: {uri['demand_mw'].max():,.0f} MW")
print(f"Peak demand datetime: {uri.loc[peak_idx, 'datetime']}")
print(f"Average baseline Feb demand: {uri['baseline_load_mw'].mean():,.0f} MW")
print(f"Peak demand surge above baseline: {uri['demand_surge_mw'].max():,.0f} MW")
print(f"Minimum demand during storm window: {uri['demand_mw'].min():,.0f} MW")
print(f"Outage-flag hours in Feb 10-20 window: {int(uri['outage_period_flag'].sum())}")
uri_facts = {'total_forced_offline_gw':34.0,'gas_offline_gw':30.0,'wind_offline_gw':2.0,'coal_offline_gw':1.5,'other_offline_gw':0.5,'peak_demand_gw':69.0,'available_generation_gw':46.0,'peak_shortfall_gw':23.0,'customers_affected':4_500_000,'hours_of_outages':78}
print('\nGeneration shortfall analysis (from ERCOT February 2021 Review / public post-storm reporting):')
print(f"  Total forced offline at peak: {uri_facts['total_forced_offline_gw']:.0f} GW")
print(f"  Gas plant freeze-offs: {uri_facts['gas_offline_gw']:.0f} GW ({uri_facts['gas_offline_gw']/uri_facts['total_forced_offline_gw']*100:.0f}% of total shortfall)")
print(f"  Wind curtailment/offline: {uri_facts['wind_offline_gw']:.0f} GW ({uri_facts['wind_offline_gw']/uri_facts['total_forced_offline_gw']*100:.0f}% of total shortfall)")
print(f"  Coal and other: {uri_facts['coal_offline_gw'] + uri_facts['other_offline_gw']:.1f} GW")
print(f"\n  CRITICAL FINDING: Gas plant failures caused {uri_facts['gas_offline_gw']/uri_facts['total_forced_offline_gw']*100:.0f}% of the generation shortfall")
print(f"  Wind turbines caused {uri_facts['wind_offline_gw']/uri_facts['total_forced_offline_gw']*100:.0f}% of the shortfall")
print('  Wind largely performed near capacity factor expectations given weather conditions')
print(f"\n  Customers affected: {uri_facts['customers_affected']:,}")
print(f"  Peak shortfall: {uri_facts['peak_shortfall_gw']:.0f} GW")


WINTER STORM URI — COMPLETE ANALYSIS
Peak demand: 69,692 MW
Peak demand datetime: 2021-02-14 20:00:00
Average baseline Feb demand: 39,540 MW
Peak demand surge above baseline: 29,990 MW
Minimum demand during storm window: 33,953 MW
Outage-flag hours in Feb 10-20 window: 203

Generation shortfall analysis (from ERCOT February 2021 Review / public post-storm reporting):
  Total forced offline at peak: 34 GW
  Gas plant freeze-offs: 30 GW (88% of total shortfall)
  Wind curtailment/offline: 2 GW (6% of total shortfall)
  Coal and other: 2.0 GW

  CRITICAL FINDING: Gas plant failures caused 88% of the generation shortfall
  Wind turbines caused 6% of the shortfall
  Wind largely performed near capacity factor expectations given weather conditions

  Customers affected: 4,500,000
  Peak shortfall: 23 GW


In [3]:

fig = go.Figure()
fig.add_trace(go.Scatter(x=uri['datetime'], y=uri['demand_mw']/1000, name='Actual Demand (GW)', line=dict(color='#E53935', width=3)))
fig.add_trace(go.Scatter(x=uri['datetime'], y=uri['baseline_load_mw']/1000, name='Normal Feb Baseline (GW)', line=dict(color='#1E88E5', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=uri['datetime'], y=uri['baseline_load_mw']/1000, fill='tonexty', fillcolor='rgba(229, 57, 53, 0.15)', line=dict(width=0), name='Demand Surge'))
fig.add_vrect(x0='2021-02-11', x1='2021-02-19', fillcolor='rgba(211, 47, 47, 0.08)', layer='below', line_width=0)
fig.add_annotation(x=pd.Timestamp('2021-02-14'), y=uri['demand_mw'].max()/1000 + 1, text=f"Peak demand: {uri['demand_mw'].max()/1000:.0f} GW<br>+{uri['demand_surge_mw'].max()/1000:.0f} GW above normal", showarrow=True, arrowhead=2, bgcolor='white', bordercolor='#E53935')
fig.add_annotation(x=pd.Timestamp('2021-02-12'), y=1.04, xref='x', yref='paper', text='Forced Outage Period', showarrow=False, font=dict(color='#B71C1C'))
fig = apply_chart_standard(fig, 'ERCOT Winter Storm Uri: Demand Surge February 2021', 'Date', 'Power (GW)')
fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02))
save_chart(fig, 'chart_3a_storm_uri_timeline')
fig.show()


Saved: outputs/charts/chart_3a_storm_uri_timeline.html


Saved: outputs/charts/chart_3a_storm_uri_timeline.png


In [4]:

fuel_failure_data = pd.DataFrame({'fuel_type':['Gas Plants\n(Freeze-offs)','Wind\n(Curtailment)','Coal & Other','Nuclear\n(Trips)'],'gw_offline':[30.0,2.0,1.5,0.5],'pct_of_shortfall':[88.2,5.9,4.4,1.5],'color':['#FF7043','#2196F3','#212121','#9C27B0']})
print('Fuel failure breakdown:'); print(fuel_failure_data.to_string(index=False))
fig = go.Figure(go.Bar(x=fuel_failure_data['fuel_type'], y=fuel_failure_data['gw_offline'], marker_color=fuel_failure_data['color'], text=[f'{v:.0f} GW<br>({p:.0f}%)' for v,p in zip(fuel_failure_data['gw_offline'], fuel_failure_data['pct_of_shortfall'])], textposition='outside', textfont=dict(size=13, color='black')))
fig = apply_chart_standard(fig, 'Winter Storm Uri: Generation Failures by Fuel Type', 'Fuel Type', 'Generation Offline at Peak (GW)', source='Source: ERCOT February 2021 Review; public post-storm reporting')
fig.update_yaxes(range=[0,35])
fig.add_annotation(x='Wind\n(Curtailment)', y=4, text='Wind performed near<br>capacity factor expectations.<br>Gas freeze-offs drove the crisis.', showarrow=False, bgcolor='#E3F2FD', bordercolor='#1E88E5', font=dict(size=10))
save_chart(fig, 'chart_3b_uri_fuel_failures')
fig.show()


Fuel failure breakdown:
                fuel_type  gw_offline  pct_of_shortfall   color
Gas Plants\n(Freeze-offs)        30.0              88.2 #FF7043
      Wind\n(Curtailment)         2.0               5.9 #2196F3
             Coal & Other         1.5               4.4 #212121
         Nuclear\n(Trips)         0.5               1.5 #9C27B0
Saved: outputs/charts/chart_3b_uri_fuel_failures.html


Saved: outputs/charts/chart_3b_uri_fuel_failures.png
